# ⏳ Ablation Study: Dual-Stream Fusion + Unidirectional LSTM (Binary Classification)

### **Objective:**
Evaluate student engagement classification using **EfficientNet-B0 + OpenFace Dual-Stream Fusion** with a standard **Unidirectional LSTM (forward temporal flow)** for temporal modeling instead of the Transformer.

| Component | Proposed Full Model (`colab_binary_transformer`) | LSTM Ablation Model (`colab_binary_lstm`) |
|---|---|---|
| **Stream 1 (Visual CNN)** | EfficientNet-B0 (1280-dim) | EfficientNet-B0 (1280-dim) |
| **Stream 2 (Facial AUs)** | OpenFace (709-dim) | OpenFace (709-dim) |
| **Fusion Layer** | Linear Projection (1989 $\rightarrow$ 128) + LayerNorm | Linear Projection (1989 $\rightarrow$ 128) + LayerNorm |
| **Temporal Modeling** | ⚡ **3-Layer Transformer Encoder** (Self-Attention) | ⏳ **2-Layer Unidirectional LSTM** (Hidden=128) |
| **Classification Head** | MLP (128 $\rightarrow$ 64 $\rightarrow$ 32 $\rightarrow$ 2) | MLP (128 $\rightarrow$ 64 $\rightarrow$ 32 $\rightarrow$ 2) |
| **Target Classes** | 2 (Engaged vs Disengaged) | 2 (Engaged vs Disengaged) |
| **Hyperparameters** | Identical (lr=5e-4, weight_decay=1e-3, dropout=0.3) | Identical (lr=5e-4, weight_decay=1e-3, dropout=0.3) |

---

### **Step 1: Mount Google Drive**
Click the **Play ▶** button. Authorize access to Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive successfully mounted!")

### **Step 2: Set Working Directory**
Navigates to the `colab_binary_lstm` directory in Google Drive.

In [ ]:
import os
import sys

possible_paths = [
    "/content/drive/MyDrive/Student_Engagement_DL/colab_binary_lstm",
    "/content/drive/MyDrive/colab_binary_lstm",
    "/content/drive/MyDrive/Binary classification/colab_binary_lstm",
    "/content/drive/Student_Engagement_DL/colab_binary_lstm",
]

working_dir = None
for p in possible_paths:
    if os.path.exists(p):
        working_dir = p
        break

if working_dir is None:
    working_dir = "/content/drive/MyDrive/Student_Engagement_DL/colab_binary_lstm"
    os.makedirs(working_dir, exist_ok=True)

if working_dir not in sys.path:
    sys.path.insert(0, working_dir)

%cd $working_dir
print("\n📁 Current working directory:", os.getcwd())
print("📄 Available files:", os.listdir('.'))

### **Step 3: Install Required Dependencies**
Installs PyTorch, torchvision, scikit-learn, pandas, matplotlib, and seaborn.

In [ ]:
!pip install -q torch torchvision scikit-learn pandas matplotlib seaborn

### **Step 4: Verify Input Data (`sequence_fuse_features_binary`)**
Checks that the preprocessed feature files exist in Google Drive and verifies the feature dimension (`1989 = 1280 + 709`).

In [ ]:
import config_binary_lstm as config
import torch

seq_dir = config.SEQUENCE_FUSED_DIR
print(f"Checking feature folder: {seq_dir}")

if os.path.exists(seq_dir):
    files = [f for f in os.listdir(seq_dir) if f.endswith('.pt')]
    print(f"✅ Successfully found {len(files)} preprocessed feature files (.pt)!")
    if len(files) > 0:
        sample = torch.load(os.path.join(seq_dir, files[0]), map_location="cpu")
        print(f"   - Sample video     : {files[0]}")
        print(f"   - Frame count      : {sample['sequence'].shape[0]}")
        print(f"   - Feature dimension: {sample['sequence'].shape[1]} (1280 CNN + 709 OpenFace)")
        print(f"   - Class label      : {sample.get('label', 'N/A')} ({config.CLASS_NAMES[sample.get('label', 0)]})")
else:
    print(f"❌ Folder not found at {seq_dir}!")

### **Step 5: Calculate & Display Parameter Count of LSTM Architecture**
Displays the parameter breakdown of EfficientNet-B0, OpenFace, LSTM layers, and Classification Head.

In [ ]:
from models_binary_lstm import BinaryEngagementLSTM, display_parameter_summary

model = BinaryEngagementLSTM()
param_info = display_parameter_summary(model)

### **Step 6: Train the Unidirectional LSTM Model**
Trains the LSTM model using AdamW, Cosine Annealing, and Early Stopping (`patience=10`). Displays loss and accuracy per epoch.

In [ ]:
!python train_binary_lstm.py

### **Step 7: Evaluate LSTM on Held-Out Test Videos**
Computes video-level Accuracy, Precision, Recall, F1-Score, Confusion Matrix, and ROC Curve.

In [ ]:
!python evaluate_binary_lstm.py

### **Step 8: Visualize Evaluation Plots & Confusion Matrix**
Displays the generated Confusion Matrix, ROC Curve, and Loss/Accuracy Curves.

In [ ]:
from IPython.display import Image, display
import config_binary_lstm as config

print("📊 1. Confusion Matrix (LSTM Ablation):")
if os.path.exists(config.CONFUSION_MATRIX_PATH):
    display(Image(filename=config.CONFUSION_MATRIX_PATH))

print("\n📈 2. ROC Curve (LSTM Ablation):")
if os.path.exists(config.ROC_CURVE_PATH):
    display(Image(filename=config.ROC_CURVE_PATH))

curves_path = os.path.join(config.EVAL_OUT_DIR, "training_curves_binary_lstm.png")
print("\n📉 3. Training & Validation Curves:")
if os.path.exists(curves_path):
    display(Image(filename=curves_path))

### **Step 9: View Video Predictions Summary**
Displays sample test predictions with confidence probabilities.

In [ ]:
import pandas as pd
import config_binary_lstm as config

if os.path.exists(config.PREDICTIONS_CSV_PATH):
    df = pd.read_csv(config.PREDICTIONS_CSV_PATH)
    display(df.head(15))
else:
    print("Predictions file not found. Run Step 7 first.")